# Mapa de Problemas em Ciclofaixas do Rio de Janeiro

**Objetivo:** Criar um mapa interativo onde qualquer pessoa pode **clicar num local e reportar um problema** na infraestrutura cicloviária.

### O que este notebook faz

1. Baixa dados de ciclovias do Rio de Janeiro (OpenStreetMap)
2. Gera um mapa interativo com camadas por tipo de ciclovia
3. **Adiciona a funcionalidade de reportar problemas:**
   - Clique no mapa para marcar um local com problema
   - Preencha: categoria, descrição, severidade
   - Anexe fotos (múltiplas)
   - Exporte/importe os dados como JSON

### Bibliotecas utilizadas

| Biblioteca | Para que serve |
|---|---|
| `osmnx` | Baixar dados geográficos do OpenStreetMap |
| `geopandas` | Manipular dados geográficos como tabelas |
| `folium` | Criar mapas interativos em HTML (baseado no Leaflet.js) |
| `branca` | Injetar HTML/CSS/JavaScript customizado no mapa |

## 1. Instalação e configuração

In [ ]:
!pip install osmnx geopandas folium

In [ ]:
import os
import osmnx as ox
import geopandas as gpd
import pandas as pd
import folium
from branca.element import Element

# Criar pastas para organizar os arquivos gerados
for pasta in ["dados", "resultados"]:
    os.makedirs(pasta, exist_ok=True)

print("Bibliotecas importadas e pastas criadas com sucesso!")

## 2. Download dos dados do OpenStreetMap

Baixamos dois tipos de infraestrutura cicloviária:

| Tag OSM | Significado |
|---|---|
| `highway=cycleway` | Via exclusiva para bicicletas |
| `cycleway=lane/track/shared_lane/shared` | Ciclofaixa ou via compartilhada em rua normal |

> **Nota:** O download pode levar de 30s a 2 minutos.

In [ ]:
cidade = "Rio de Janeiro, Brazil"

# === FONTE 1: Ciclovias exclusivas (highway=cycleway) ===
print(f"Baixando ciclovias exclusivas de '{cidade}'...")
ciclovias_exclusivas = ox.features_from_place(cidade, tags={"highway": "cycleway"})
ciclovias_exclusivas = ciclovias_exclusivas[
    ciclovias_exclusivas.geometry.geom_type == "LineString"
].copy()
ciclovias_exclusivas["tipo_ciclovia"] = "Ciclovia Exclusiva"
print(f"  Encontradas: {len(ciclovias_exclusivas)} vias")

# === FONTE 2: Ciclofaixas em ruas normais ===
print(f"\nBaixando ciclofaixas em vias normais...")
ciclofaixas = ox.features_from_place(cidade, tags={"cycleway": True})
ciclofaixas = ciclofaixas[
    ciclofaixas.geometry.geom_type == "LineString"
].copy()

# Filtrar apenas os que realmente têm infraestrutura
valores_validos = ["lane", "track", "shared_lane", "shared", "yes"]
ciclofaixas = ciclofaixas[
    ciclofaixas["cycleway"].apply(
        lambda x: str(x) if isinstance(x, list) else x
    ).isin(valores_validos)
].copy()

def classificar_ciclofaixa(valor):
    """Classifica o tipo de ciclofaixa baseado na tag do OpenStreetMap."""
    valor = str(valor)
    if valor in ["lane", "track"]:
        return "Ciclofaixa Dedicada"
    else:
        return "Via Compartilhada"

ciclofaixas["tipo_ciclovia"] = ciclofaixas["cycleway"].apply(classificar_ciclofaixa)
print(f"  Encontradas: {len(ciclofaixas)} vias")

# === COMBINAR AS DUAS FONTES ===
colunas = ["geometry", "tipo_ciclovia", "name"]
for col in colunas:
    if col not in ciclovias_exclusivas.columns:
        ciclovias_exclusivas[col] = None
    if col not in ciclofaixas.columns:
        ciclofaixas[col] = None

infraestrutura_bike = gpd.GeoDataFrame(
    pd.concat([ciclovias_exclusivas[colunas], ciclofaixas[colunas]], ignore_index=True),
    geometry="geometry",
    crs=ciclovias_exclusivas.crs,
)

print(f"\n{'='*50}")
print(f"TOTAL: {len(infraestrutura_bike)} trechos de infraestrutura cicloviária")
print(f"{'='*50}")

## 3. Exploração rápida dos dados

In [ ]:
# Quantas vias de cada tipo?
print("Quantidade de vias por tipo:")
print(infraestrutura_bike["tipo_ciclovia"].value_counts())

# Simplificar geometrias para performance
infraestrutura_bike["geometry"] = infraestrutura_bike["geometry"].simplify(tolerance=0.0005)
print(f"\nGeometrias simplificadas para melhor performance no navegador.")

## 4. Funcionalidade de reportar problemas

Aqui definimos todo o **JavaScript e CSS** que será injetado no mapa para permitir reportar problemas.

### Como funciona

- **Clique no mapa** para abrir o formulário de reporte
- Preencha a **categoria**, **descrição**, **severidade** e anexe **fotos**
- Os dados ficam salvos no **localStorage** do navegador (persistem entre sessões)
- Use os botões no painel para **exportar** ou **importar** dados em JSON

### Categorias de problemas

| Categoria | Exemplos |
|---|---|
| Buraco/Desnível | Piso danificado, buraco, desnível perigoso |
| Falta de Sinalização | Sem placa, pintura apagada |
| Obstrução | Objeto ou veículo bloqueando a ciclovia |
| Trecho Perigoso | Cruzamento sem proteção, curva sem visibilidade |
| Iluminação | Falta de iluminação, poste apagado |
| Outro | Problema não categorizado |

### Níveis de severidade

| Severidade | Cor no mapa | Significado |
|---|---|---|
| Info | Azul | Sugestão de melhoria, observação |
| Baixo | Verde | Problema menor, não impede o uso |
| Médio | Laranja | Problema significativo, requer atenção |
| Crítico | Vermelho | Perigo real, risco de acidente |

In [ ]:
# ============================================================
# JavaScript e CSS para a funcionalidade de reportar problemas
# ============================================================
# Este bloco define todo o código frontend que será injetado
# no HTML do mapa. Não precisa entender cada linha de JS —
# o importante é entender a ESTRUTURA (veja comentários abaixo).

JAVASCRIPT_PROBLEMAS = """
<style>
/* ========== ESTILOS DO MODAL DE REPORTE ========== */

/* Fundo escurecido atrás do modal */
#modal-overlay {
    display: none;
    position: fixed;
    top: 0; left: 0;
    width: 100%; height: 100%;
    background: rgba(0,0,0,0.5);
    z-index: 10000;
    justify-content: center;
    align-items: center;
}

/* Caixa do formulário */
#modal-form {
    background: white;
    border-radius: 12px;
    padding: 24px;
    width: 420px;
    max-height: 85vh;
    overflow-y: auto;
    box-shadow: 0 8px 32px rgba(0,0,0,0.3);
    font-family: 'Segoe UI', Arial, sans-serif;
}

#modal-form h3 {
    margin: 0 0 16px 0;
    color: #333;
    font-size: 18px;
}

#modal-form label {
    display: block;
    margin: 12px 0 4px 0;
    font-weight: 600;
    color: #555;
    font-size: 13px;
}

#modal-form select,
#modal-form textarea {
    width: 100%;
    padding: 8px 10px;
    border: 1px solid #ccc;
    border-radius: 6px;
    font-size: 14px;
    box-sizing: border-box;
}

#modal-form textarea {
    height: 80px;
    resize: vertical;
}

/* Área de preview das imagens selecionadas */
#img-preview {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    margin-top: 8px;
}

#img-preview img {
    width: 70px;
    height: 70px;
    object-fit: cover;
    border-radius: 6px;
    border: 1px solid #ddd;
    cursor: pointer;
}

/* Botões do formulário */
.modal-buttons {
    display: flex;
    gap: 10px;
    margin-top: 18px;
}

.btn-salvar {
    flex: 1;
    padding: 10px;
    background: #2196F3;
    color: white;
    border: none;
    border-radius: 6px;
    font-size: 14px;
    font-weight: 600;
    cursor: pointer;
}

.btn-salvar:hover { background: #1976D2; }

.btn-cancelar {
    flex: 1;
    padding: 10px;
    background: #eee;
    color: #333;
    border: none;
    border-radius: 6px;
    font-size: 14px;
    cursor: pointer;
}

.btn-cancelar:hover { background: #ddd; }

/* ========== PAINEL DE CONTROLE (canto inferior direito) ========== */
#painel-controle {
    position: fixed;
    bottom: 20px;
    right: 20px;
    z-index: 9999;
    background: white;
    border-radius: 10px;
    padding: 14px;
    box-shadow: 0 2px 12px rgba(0,0,0,0.2);
    font-family: 'Segoe UI', Arial, sans-serif;
    font-size: 13px;
    min-width: 200px;
}

#painel-controle h4 {
    margin: 0 0 10px 0;
    font-size: 14px;
    color: #333;
}

#painel-controle button {
    display: block;
    width: 100%;
    padding: 8px;
    margin: 5px 0;
    border: 1px solid #ccc;
    border-radius: 6px;
    background: #f8f8f8;
    cursor: pointer;
    font-size: 13px;
}

#painel-controle button:hover { background: #e8e8e8; }

/* ========== LIGHTBOX (visualizar imagem em tela cheia) ========== */
#lightbox {
    display: none;
    position: fixed;
    top: 0; left: 0;
    width: 100%; height: 100%;
    background: rgba(0,0,0,0.85);
    z-index: 20000;
    justify-content: center;
    align-items: center;
    cursor: pointer;
}

#lightbox img {
    max-width: 90%;
    max-height: 90%;
    border-radius: 8px;
}

/* ========== INSTRUÇÃO FLUTUANTE ========== */
#instrucao-clique {
    position: fixed;
    top: 12px;
    left: 50%;
    transform: translateX(-50%);
    z-index: 9999;
    background: #2196F3;
    color: white;
    padding: 10px 20px;
    border-radius: 8px;
    font-family: 'Segoe UI', Arial, sans-serif;
    font-size: 14px;
    font-weight: 600;
    box-shadow: 0 2px 8px rgba(0,0,0,0.2);
    pointer-events: none;
}
</style>

<!-- Instrução flutuante no topo -->
<div id=\"instrucao-clique\">Clique no mapa para reportar um problema</div>

<!-- Modal de reporte -->
<div id=\"modal-overlay\">
  <div id=\"modal-form\">
    <h3>Reportar Problema</h3>
    <p id=\"modal-coords\" style=\"color:#888; font-size:12px; margin:0 0 8px 0;\"></p>

    <label for=\"sel-categoria\">Categoria</label>
    <select id=\"sel-categoria\">
      <option value=\"Buraco/Desnivel\">Buraco / Desnivel</option>
      <option value=\"Falta de Sinalizacao\">Falta de Sinalizacao</option>
      <option value=\"Obstrucao\">Obstrucao</option>
      <option value=\"Trecho Perigoso\">Trecho Perigoso</option>
      <option value=\"Iluminacao\">Iluminacao</option>
      <option value=\"Outro\">Outro</option>
    </select>

    <label for=\"txt-descricao\">Descricao</label>
    <textarea id=\"txt-descricao\" placeholder=\"Descreva o problema...\"></textarea>

    <label for=\"sel-severidade\">Severidade</label>
    <select id=\"sel-severidade\">
      <option value=\"Info\">Info (sugestao/observacao)</option>
      <option value=\"Baixo\">Baixo (problema menor)</option>
      <option value=\"Medio\" selected>Medio (requer atencao)</option>
      <option value=\"Critico\">Critico (risco de acidente)</option>
    </select>

    <label for=\"input-fotos\">Fotos (opcional, multiplas)</label>
    <input type=\"file\" id=\"input-fotos\" accept=\"image/*\" multiple
           style=\"font-size:13px; margin-top:4px;\">
    <div id=\"img-preview\"></div>

    <div class=\"modal-buttons\">
      <button class=\"btn-salvar\" onclick=\"salvarProblema()\">Salvar</button>
      <button class=\"btn-cancelar\" onclick=\"fecharModal()\">Cancelar</button>
    </div>
  </div>
</div>

<!-- Painel de controle -->
<div id=\"painel-controle\">
  <h4>Problemas Reportados</h4>
  <div id=\"contador\">0 problemas</div>
  <button onclick=\"exportarJSON()\">Exportar JSON</button>
  <button onclick=\"document.getElementById('input-importar').click()\">Importar JSON</button>
  <input type=\"file\" id=\"input-importar\" accept=\".json\" style=\"display:none\"
         onchange=\"importarJSON(event)\">
  <button onclick=\"limparTodos()\" style=\"color:#c62828; border-color:#c62828;\">Limpar Tudo</button>
</div>

<!-- Lightbox para ver imagem em tela cheia -->
<div id=\"lightbox\" onclick=\"this.style.display='none'\">
  <img id=\"lightbox-img\" src=\"\">
</div>

<script>
// ============================================================
//  SISTEMA DE REPORTE DE PROBLEMAS EM CICLOFAIXAS
// ============================================================
// Este script gerencia:
//   1. Clique no mapa -> abre formulario
//   2. Upload de imagens -> converte para base64
//   3. Salva no localStorage do navegador
//   4. Exporta/importa como arquivo JSON
//   5. Marcadores coloridos por severidade
// ============================================================

// --- Configuracao ---
var STORAGE_KEY = 'problemas_ciclovia_rj';

// Cores dos marcadores por nivel de severidade
var CORES_SEVERIDADE = {
    'Info':    '#2196F3',  // azul
    'Baixo':   '#4CAF50',  // verde
    'Medio':   '#FF9800',  // laranja
    'Critico': '#F44336'   // vermelho
};

// --- Estado global ---
var problemas = [];           // lista de todos os problemas
var marcadores = [];          // referencia aos marcadores no mapa (para poder remover)
var clickLat = 0;             // coordenadas do ultimo clique
var clickLng = 0;
var imagensBase64 = [];       // imagens do formulario atual
var proximoId = 1;            // ID auto-incremento

// --- Referencia ao mapa Leaflet ---
// Folium cria o mapa numa variavel global. Precisamos encontra-la.
var mapaLeaflet = null;
(function() {
    // Folium armazena o mapa em uma variavel cujo nome comeca com 'map_'
    for (var nome in window) {
        if (nome.indexOf('map_') === 0 && window[nome] instanceof L.Map) {
            mapaLeaflet = window[nome];
            break;
        }
    }
    if (!mapaLeaflet) {
        console.error('Nao foi possivel encontrar o mapa Leaflet.');
        return;
    }

    // Registrar evento de clique no mapa
    mapaLeaflet.on('click', function(e) {
        clickLat = e.latlng.lat;
        clickLng = e.latlng.lng;
        abrirModal(clickLat, clickLng);
    });

    // Carregar problemas salvos do localStorage
    carregarDoStorage();
})();

// ============================================================
//  FUNCOES DO MODAL (formulario de reporte)
// ============================================================

function abrirModal(lat, lng) {
    // Exibir coordenadas no formulario
    document.getElementById('modal-coords').textContent =
        'Latitude: ' + lat.toFixed(6) + ', Longitude: ' + lng.toFixed(6);

    // Limpar formulario
    document.getElementById('sel-categoria').selectedIndex = 0;
    document.getElementById('txt-descricao').value = '';
    document.getElementById('sel-severidade').value = 'Medio';
    document.getElementById('input-fotos').value = '';
    document.getElementById('img-preview').innerHTML = '';
    imagensBase64 = [];

    // Mostrar modal
    var overlay = document.getElementById('modal-overlay');
    overlay.style.display = 'flex';
}

function fecharModal() {
    document.getElementById('modal-overlay').style.display = 'none';
    imagensBase64 = [];
}

// ============================================================
//  UPLOAD DE IMAGENS (converte para base64 no navegador)
// ============================================================

document.getElementById('input-fotos').addEventListener('change', function(e) {
    var arquivos = e.target.files;
    var preview = document.getElementById('img-preview');
    preview.innerHTML = '';
    imagensBase64 = [];

    for (var i = 0; i < arquivos.length; i++) {
        (function(arquivo) {
            var reader = new FileReader();
            reader.onload = function(ev) {
                // Comprimir a imagem para nao estourar o localStorage
                comprimirImagem(ev.target.result, 800, 0.7, function(comprimida) {
                    imagensBase64.push(comprimida);

                    // Mostrar preview da imagem
                    var img = document.createElement('img');
                    img.src = comprimida;
                    img.title = arquivo.name;
                    img.onclick = function() { abrirLightbox(comprimida); };
                    preview.appendChild(img);
                });
            };
            reader.readAsDataURL(arquivo);
        })(arquivos[i]);
    }
});

// Redimensiona e comprime a imagem usando canvas
// maxLado: largura maxima em pixels
// qualidade: 0.0 a 1.0 (compressao JPEG)
function comprimirImagem(dataUrl, maxLado, qualidade, callback) {
    var img = new Image();
    img.onload = function() {
        var largura = img.width;
        var altura = img.height;

        // Redimensionar se for maior que maxLado
        if (largura > maxLado || altura > maxLado) {
            if (largura > altura) {
                altura = Math.round(altura * maxLado / largura);
                largura = maxLado;
            } else {
                largura = Math.round(largura * maxLado / altura);
                altura = maxLado;
            }
        }

        var canvas = document.createElement('canvas');
        canvas.width = largura;
        canvas.height = altura;
        var ctx = canvas.getContext('2d');
        ctx.drawImage(img, 0, 0, largura, altura);

        callback(canvas.toDataURL('image/jpeg', qualidade));
    };
    img.src = dataUrl;
}

// ============================================================
//  SALVAR PROBLEMA
// ============================================================

function salvarProblema() {
    var categoria = document.getElementById('sel-categoria').value;
    var descricao = document.getElementById('txt-descricao').value.trim();
    var severidade = document.getElementById('sel-severidade').value;

    if (!descricao) {
        alert('Por favor, descreva o problema.');
        return;
    }

    var problema = {
        id: proximoId++,
        lat: clickLat,
        lng: clickLng,
        categoria: categoria,
        descricao: descricao,
        severidade: severidade,
        imagens: imagensBase64.slice(),  // copia do array
        data_criacao: new Date().toISOString()
    };

    problemas.push(problema);
    adicionarMarcador(problema);
    salvarNoStorage();
    atualizarContador();
    fecharModal();
}

// ============================================================
//  MARCADORES NO MAPA
// ============================================================

function adicionarMarcador(problema) {
    var cor = CORES_SEVERIDADE[problema.severidade] || '#999';

    // Criar icone circular colorido
    var icone = L.divIcon({
        className: '',
        html: '<div style=\"' +
              'background:' + cor + ';' +
              'width:22px; height:22px;' +
              'border-radius:50%;' +
              'border:3px solid white;' +
              'box-shadow:0 2px 6px rgba(0,0,0,0.4);' +
              '\"></div>',
        iconSize: [22, 22],
        iconAnchor: [11, 11]
    });

    // Montar HTML do popup com informacoes do problema
    var htmlPopup = '<div style=\"font-family:Segoe UI,Arial; min-width:200px; max-width:300px;\">';
    htmlPopup += '<b style=\"font-size:14px;\">' + problema.categoria + '</b>';
    htmlPopup += '<span style=\"float:right; padding:2px 8px; border-radius:4px; font-size:11px; color:white; background:' + cor + ';\">' + problema.severidade + '</span>';
    htmlPopup += '<p style=\"margin:8px 0; font-size:13px; color:#555;\">' + problema.descricao + '</p>';

    // Thumbnails das imagens (clicaveis)
    if (problema.imagens && problema.imagens.length > 0) {
        htmlPopup += '<div style=\"display:flex; flex-wrap:wrap; gap:4px; margin:8px 0;\">';
        for (var i = 0; i < problema.imagens.length; i++) {
            htmlPopup += '<img src=\"' + problema.imagens[i] + '\" ' +
                         'style=\"width:60px; height:60px; object-fit:cover; border-radius:4px; cursor:pointer;\" ' +
                         'onclick=\"abrirLightbox(this.src)\">';
        }
        htmlPopup += '</div>';
    }

    // Data e botao de remover
    var data = new Date(problema.data_criacao);
    htmlPopup += '<div style=\"font-size:11px; color:#999; margin-top:6px;\">' +
                 data.toLocaleDateString('pt-BR') + ' ' + data.toLocaleTimeString('pt-BR', {hour:'2-digit', minute:'2-digit'}) +
                 '</div>';
    htmlPopup += '<button onclick=\"removerProblema(' + problema.id + ')\" ' +
                 'style=\"margin-top:8px; padding:4px 12px; background:#f44336; color:white; border:none; border-radius:4px; cursor:pointer; font-size:12px;\">' +
                 'Remover</button>';
    htmlPopup += '</div>';

    var marcador = L.marker([problema.lat, problema.lng], {icon: icone})
        .bindPopup(htmlPopup, {maxWidth: 320})
        .addTo(mapaLeaflet);

    // Guardar referencia para poder remover depois
    marcador._problemaId = problema.id;
    marcadores.push(marcador);
}

function removerProblema(id) {
    // Remover da lista
    problemas = problemas.filter(function(p) { return p.id !== id; });

    // Remover marcador do mapa
    for (var i = marcadores.length - 1; i >= 0; i--) {
        if (marcadores[i]._problemaId === id) {
            mapaLeaflet.removeLayer(marcadores[i]);
            marcadores.splice(i, 1);
            break;
        }
    }

    salvarNoStorage();
    atualizarContador();
}

// ============================================================
//  PERSISTENCIA (localStorage)
// ============================================================

function salvarNoStorage() {
    try {
        localStorage.setItem(STORAGE_KEY, JSON.stringify(problemas));
    } catch(e) {
        alert('Erro ao salvar: o armazenamento do navegador pode estar cheio. ' +
              'Exporte os dados como JSON e limpe o historico.');
    }
}

function carregarDoStorage() {
    var dados = localStorage.getItem(STORAGE_KEY);
    if (!dados) return;

    try {
        problemas = JSON.parse(dados);
        // Recalcular proximo ID
        proximoId = 1;
        for (var i = 0; i < problemas.length; i++) {
            if (problemas[i].id >= proximoId) {
                proximoId = problemas[i].id + 1;
            }
            adicionarMarcador(problemas[i]);
        }
        atualizarContador();
    } catch(e) {
        console.error('Erro ao carregar dados salvos:', e);
    }
}

function atualizarContador() {
    var texto = problemas.length + ' problema' + (problemas.length !== 1 ? 's' : '');
    document.getElementById('contador').textContent = texto;
}

// ============================================================
//  EXPORTAR / IMPORTAR JSON
// ============================================================

function exportarJSON() {
    if (problemas.length === 0) {
        alert('Nenhum problema para exportar.');
        return;
    }

    var exportacao = {
        versao: '1.0',
        exportado_em: new Date().toISOString(),
        total: problemas.length,
        problemas: problemas
    };

    var json = JSON.stringify(exportacao, null, 2);
    var blob = new Blob([json], {type: 'application/json'});
    var url = URL.createObjectURL(blob);

    // Criar link de download invisivel e clicar nele
    var link = document.createElement('a');
    link.href = url;
    link.download = 'problemas_ciclovia_' +
                    new Date().toISOString().slice(0,10) + '.json';
    link.click();

    URL.revokeObjectURL(url);
}

function importarJSON(event) {
    var arquivo = event.target.files[0];
    if (!arquivo) return;

    var reader = new FileReader();
    reader.onload = function(e) {
        try {
            var dados = JSON.parse(e.target.result);
            var lista = dados.problemas || dados;  // aceita ambos os formatos

            if (!Array.isArray(lista)) {
                alert('Formato de arquivo invalido.');
                return;
            }

            var importados = 0;
            for (var i = 0; i < lista.length; i++) {
                var p = lista[i];
                // Verificar campos minimos
                if (p.lat && p.lng && p.categoria && p.descricao) {
                    p.id = proximoId++;
                    problemas.push(p);
                    adicionarMarcador(p);
                    importados++;
                }
            }

            salvarNoStorage();
            atualizarContador();
            alert(importados + ' problema(s) importado(s) com sucesso!');
        } catch(err) {
            alert('Erro ao ler o arquivo: ' + err.message);
        }
    };
    reader.readAsText(arquivo);

    // Limpar input para permitir reimportar o mesmo arquivo
    event.target.value = '';
}

// ============================================================
//  UTILIDADES
// ============================================================

function limparTodos() {
    if (problemas.length === 0) {
        alert('Nenhum problema para limpar.');
        return;
    }

    if (!confirm('Tem certeza que deseja remover todos os ' + problemas.length + ' problemas? Esta acao nao pode ser desfeita.')) {
        return;
    }

    // Remover todos os marcadores do mapa
    for (var i = 0; i < marcadores.length; i++) {
        mapaLeaflet.removeLayer(marcadores[i]);
    }
    marcadores = [];
    problemas = [];
    proximoId = 1;

    salvarNoStorage();
    atualizarContador();
}

function abrirLightbox(src) {
    document.getElementById('lightbox-img').src = src;
    document.getElementById('lightbox').style.display = 'flex';
}
</script>
"""

print(f"JavaScript definido: {len(JAVASCRIPT_PROBLEMAS)} caracteres")
print("Pronto para injetar no mapa!")

## 5. Gerar o mapa com a funcionalidade de reportar problemas

Aqui criamos o mapa final: camadas de ciclovias + formulário de reporte.

In [ ]:
# Cores das camadas de ciclovias
cores_por_tipo = {
    "Ciclovia Exclusiva":   "red",
    "Ciclofaixa Dedicada":  "blue",
    "Via Compartilhada":    "orange",
}

# Criar mapa base centrado no Rio de Janeiro
mapa_rj = folium.Map(
    location=[-22.9068, -43.1729],
    zoom_start=12,
    tiles="CartoDB positron",
)

# Adicionar camadas de ciclovias
for tipo, cor in cores_por_tipo.items():
    subset = infraestrutura_bike[infraestrutura_bike["tipo_ciclovia"] == tipo]
    if subset.empty:
        continue

    folium.GeoJson(
        subset,
        name=tipo,
        style_function=lambda x, c=cor: {"color": c, "weight": 2.5},
        tooltip=folium.GeoJsonTooltip(
            fields=["name", "tipo_ciclovia"],
            aliases=["Nome da via:", "Tipo:"],
        ),
    ).add_to(mapa_rj)
    print(f"  Camada '{tipo}': {len(subset)} trechos (cor: {cor})")

# Adicionar controle de camadas
folium.LayerControl().add_to(mapa_rj)

# --- INJETAR O JAVASCRIPT DE REPORTAR PROBLEMAS ---
# Isso adiciona o formulario, painel de controle e toda a logica
# diretamente no HTML gerado pelo Folium
mapa_rj.get_root().html.add_child(Element(JAVASCRIPT_PROBLEMAS))

# Salvar como HTML
arquivo_mapa = os.path.join("resultados", "mapa_problemas_ciclovias.html")
mapa_rj.save(arquivo_mapa)

print(f"\nMapa salvo em: {arquivo_mapa}")
print("Abra esse arquivo no navegador para usar!")

## 6. Como usar o mapa

### Instrucoes para os estagiarios

1. **Abra o arquivo** `resultados/mapa_problemas_ciclovias.html` no navegador (Chrome, Firefox, Edge)
2. **Clique no mapa** no local onde ha um problema
3. **Preencha o formulario:**
   - Escolha a categoria do problema
   - Escreva uma descricao
   - Defina a severidade
   - Anexe fotos (opcional, pode ser mais de uma)
4. **Clique em Salvar** — o marcador aparece no mapa
5. **Os dados ficam salvos** no navegador (localStorage) — persistem entre sessoes

### Exportar e compartilhar dados

- Clique em **"Exportar JSON"** no painel inferior direito
- O arquivo `problemas_ciclovia_YYYY-MM-DD.json` sera baixado
- Envie esse arquivo para o professor ou colegas
- Para carregar dados de outra pessoa, use **"Importar JSON"**

### Dicas

- Clique num marcador existente para ver detalhes e fotos
- Clique numa foto para abrir em tela cheia
- Use o botao **"Remover"** no popup para apagar um problema
- Use **"Limpar Tudo"** para recomecar do zero

### Limitacoes

- As fotos sao comprimidas para 800px e JPEG 70% (para caber no localStorage)
- O localStorage tem limite de ~5-10 MB por site — suficiente para ~50-100 problemas com fotos
- Se o armazenamento encher, exporte os dados como JSON e use "Limpar Tudo"

In [ ]:
# Visualizar o mapa inline (funciona no Jupyter/Colab)
# Nota: a funcionalidade de reporte so funciona 100% no HTML salvo,
# pois o iframe do Jupyter limita acesso ao localStorage.
mapa_rj